**Laboratorio de métodos cuantitativos aplicados a la Gestión**

---


# **Clase 5 - Matrices en Python**

## Complemento

Este notebook se complementa con la presentación: **Algebra_Data_v2.pdf**

Te recomendamos leer el PDF para trabajar con este notebook y tener una mejor comprensión de los conceptos teóricos.

¿Qué vamos a poder hacer?

🔍 Detectar y filtrar información relevante, como productos con bajo stock, categorías más frecuentes o proveedores más importantes.

🧮 Realizar operaciones con matrices en Python y entender cómo aplicarlas en contextos reales.

🧩 Construir e interpretar el modelo insumo-producto de Leontief, entendiendo cómo interactúan los sectores productivos de una economía.

🏗️ Diseñar y resolver problemas organizacionales simples a partir de data estructurada.

## Objetivos de la clase

- Entender por qué NumPy existe y cuándo conviene sobre una lista de Python
- Operar con vectores y matrices **sin escribir un solo bucle** (vectorización)
- Leer una tabla como matriz: `shape`, filas, columnas y el famoso `axis`
- Distinguir el producto **elemento a elemento** del **producto matricial**
- Reconocer qué modela cada estructura especial (diagonal, triangular, simétrica)

In [1]:
!pip install numpy matplotlib pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [2]:
#importamos las librerías que vamos a usar en esta ocasión
import numpy as np
import pandas as pd



---
## ⚡ Vectorización: la razón de ser de NumPy

Supongamos que hay que aplicarle el IVA a una lista de precios. Hay tres formas de hacerlo, y las
tres dan el mismo resultado:

| Cómo | Código |
|---|---|
| Con un **bucle** | `for p in precios: nuevos.append(round(p * 1.21, 2))` |
| Con una **comprensión** | `[round(p * 1.21, 2) for p in precios]` |
| Con **NumPy** | `np.round(precios * 1.21, 2)` |

La tercera no solo es más corta: es **mucho más rápida**, porque la multiplicación ocurre de una vez
sobre todo el bloque de memoria en lugar de elemento por elemento.

Eso se llama **vectorización**, y es la idea central de toda la materia: *decile qué querés hacer
con todos los datos, no cómo recorrerlos*.

In [ ]:
import numpy as np

precios_lista = [12400, 19900, 7450, 4800, 14500, 9900]   # una lista común de Python
precios = np.array(precios_lista)                          # el mismo dato, como array de NumPy

print("Lista de Python :", precios_lista)
print("Array de NumPy  :", precios)

# La diferencia aparece al operar:
print("\nLista * 2  →", precios_lista * 2)      # ⚠️ REPITE la lista, no multiplica
print("Array * 2  →", precios * 2)                # ✅ multiplica cada elemento

> ⚠️ **Ojo con esa diferencia.** `lista * 2` **duplica la lista** (la pega dos veces);
> `array * 2` multiplica cada valor por dos. No da error: **da otra cosa.** Es una de las confusiones
> más frecuentes al empezar.

In [ ]:
unidades = np.array([420, 145, 610, 480, 190, 130])

# Operar entre dos arrays: se aplica posición por posición
facturacion = unidades * precios

print("Unidades   :", unidades)
print("Precios    :", precios)
print("Facturación:", facturacion)
print("\nTotal facturado: $", facturacion.sum())

### `dtype`: un array es homogéneo

A diferencia de una lista, **todos los elementos de un array tienen que ser del mismo tipo**.
NumPy elige el tipo que sirva para todos, y ahí hay una trampa:

In [ ]:
enteros = np.array([420, 145, 610])
print("Solo enteros      :", enteros.dtype)

mezcla_decimal = np.array([420, 145.5, 610])
print("Con un decimal    :", mezcla_decimal.dtype, "  ← todo pasó a float")

mezcla_texto = np.array([12400, 19900, "sin dato"])
print("Con un texto      :", mezcla_texto.dtype, " ← ⚠️ TODO se volvió texto")
print("Los números ahora son:", mezcla_texto)

**Eso último rompe todo lo que venga después.** Si en una columna de precios se coló un
`"sin dato"`, `"N/D"` o un guion, el array entero pasa a ser texto y cualquier cuenta falla.

> 🎯 Es exactamente el mismo problema que veíamos con `dtype: object` en pandas. **El origen suele
> ser un solo dato mal cargado entre miles.**

---
## 🧭 `axis`: la pregunta que confunde a todo el mundo

Cuando una matriz representa una tabla, sumar tiene **dos significados distintos** y hay que elegir:

```
                 Neuquén  Bariloche  Trelew        axis=1 →  suma la FILA
   A101 Harina     420       240      180    →  840          (total del producto)
   B310 Gaseosa    610       380      250    → 1240
                    ↓         ↓        ↓
   axis=0 ↓       1030       620      430          axis=0 →  suma la COLUMNA
                                                             (total de la sucursal)
```

**La regla para no equivocarse:** `axis` indica **qué eje desaparece**.

| Código | Qué eje colapsa | Qué queda | Pregunta que responde |
|---|---|---|---|
| `matriz.sum(axis=0)` | Las filas | Un valor **por columna** | ¿Cuánto vendió cada sucursal? |
| `matriz.sum(axis=1)` | Las columnas | Un valor **por fila** | ¿Cuánto se vendió de cada producto? |
| `matriz.sum()` | Los dos | Un solo número | ¿Cuál es el total general? |

In [ ]:
# Filas = productos, columnas = sucursales
ventas = np.array([[420, 240, 180],     # A101 Harina
                   [610, 380, 250],     # B310 Gaseosa
                   [190,   0,  95]])    # L150 Detergente

productos_nom = ["A101 Harina", "B310 Gaseosa", "L150 Detergente"]
sucursales = ["Neuquén", "Bariloche", "Trelew"]

print("shape:", ventas.shape, " → (filas, columnas) = (productos, sucursales)")
print()
print("Total por SUCURSAL (axis=0):", ventas.sum(axis=0))
print("Total por PRODUCTO (axis=1):", ventas.sum(axis=1))
print("Total general             :", ventas.sum())

In [ ]:
# Lo mismo, presentado como se lo mandarías al gerente
import pandas as pd

tablero = pd.DataFrame(ventas, index=productos_nom, columns=sucursales)
tablero["TOTAL producto"] = ventas.sum(axis=1)
tablero.loc["TOTAL sucursal"] = list(ventas.sum(axis=0)) + [ventas.sum()]

tablero

---
## ✖️ `*` contra `@`: dos multiplicaciones distintas

Esta distinción **entra en el parcial** y es la fuente de errores más silenciosa de la unidad.

| Operador | Qué hace | Requisito | Resultado |
|---|---|---|---|
| `A * B` | Multiplica **celda por celda** | Misma forma | Una matriz de la misma forma |
| `A @ B` | **Producto matricial** (filas × columnas) | Columnas de A = filas de B | Una matriz nueva |

El peligro: si las formas coinciden, **`*` no da error, da otro número**.

In [ ]:
precios_prod = np.array([12400, 7450, 9900])     # un precio por producto

# ¿Cuánto facturó cada sucursal? Hay que multiplicar precios por unidades y sumar por producto.
# Eso es EXACTAMENTE un producto matricial:
facturacion_sucursal = precios_prod @ ventas

print("Facturación por sucursal:", facturacion_sucursal)
for s, f in zip(sucursales, facturacion_sucursal):
    print(f"  {s:12} $ {f:>10,.0f}")

In [ ]:
# Verificamos a mano la primera sucursal, para creerle al @
neuquen = 12400*420 + 7450*610 + 9900*190

print("Neuquén calculado a mano :", neuquen)
print("Neuquén con @            :", facturacion_sucursal[0])
print("¿Coinciden?", neuquen == facturacion_sucursal[0])

### Transponer: dar vuelta la tabla

`A.T` intercambia filas por columnas. Sirve cuando la matriz está orientada al revés de lo que
necesita la cuenta — algo que pasa **todo el tiempo** al combinar tablas de distintos sistemas.

In [ ]:
print("Original (productos × sucursales):", ventas.shape)
print(ventas)
print("\nTranspuesta (sucursales × productos):", ventas.T.shape)
print(ventas.T)

# Matrices

In [3]:
#así armamos una matriz

A = np.array([[1, 4],
              [0, 6]]) #notemos que cada fila está encerrada en corchetes
I= np.identity(2)
print(A)
print(I)


[[1 4]
 [0 6]]
[[1. 0.]
 [0. 1.]]


In [4]:
# Vector columna (3x1) — como los del ejemplo
p = np.array([[100], [250], [80]])
print(p.shape)  # (3, 1)

# Vector fila (1x3)
p_fila = np.array([[100, 250, 80]])
print(p_fila.shape)  # (1, 3)

(3, 1)
(1, 3)


In [5]:
#algunas operaciones básicas de numpy con matrices

# Suma de matrices
C = A + A

# Multiplicación de matrices
D = A @ A

# Transpuesta
A_T = A.T

# Inversa (si existe)
A_inv = np.linalg.inv(A)

print("A+A:")
print(C)
print("\n------------------------\n")

print("A x A")
print(D)
print("\n------------------------\n")

print("Transpuesta de A")
print(A_T)


A+A:
[[ 2  8]
 [ 0 12]]

------------------------

A x A
[[ 1 28]
 [ 0 36]]

------------------------

Transpuesta de A
[[1 0]
 [4 6]]


In [6]:
#algunas características de las matrices
#orden - dimensiones de la matriz (filas, columnas)
print(A.shape)
#rango - número de filas/columnas linealmente independientes
rango_A=np.linalg.matrix_rank(A)
print(rango_A)
#traza - suma de elementos de la diagonal principal
print(np.trace(A))

(2, 2)
2
7


In [7]:
R= np.array([[0,1],
             [0,1]])


In [8]:
#calculemos determinantes
det_R=np.linalg.det(R)
det_A=np.linalg.det(A)
print(det_R)
print(det_A)


0.0
6.0


In [9]:
C=np.array([[3, 4, 5],
            [6, 7, 8],
            [3, 9, 1]])
det_c=np.linalg.det(C)
print(det_c)

41.999999999999986


Definimos una función que nos diga si dos matrices son multiplicables

In [10]:
def son_multiplicables(matriz1, matriz2):

    "Recibe dos matrices (listas de listas) y devuelve True si se pueden multiplicar"


    # Para multiplicar dos matrices, el número de columnas de la primera debe ser igual al número de filas de la segunda.
    columnasm1 = len(matriz1[0])
    filasm2 = len(matriz2)

    return columnasm1 == filasm2


In [11]:
#la probamos con un par de matrices
son_multiplicables(A,R)

True

# DataFrame con Pandas

💪 Pandas nos permite ver todo más bonito y también tiene funciones que facilitan el análisis descriptivo de los datos. Vamos a importar una base de datos sencilla para usar un poco esa librería

En términos generales, muchos conceptos que conocemos de **Numpy** se repiten. Ejemplos de esto incluyen el **indexado** y el **corte (slicing)**, pero también la forma en que se llaman funciones sobre los datos almacenados en un DataFrame.

Para dar un solo ejemplo, si probamos el siguiente código en el DataFrame de abajo:

```python
df["Ventas mensuales"].mean()
```

Es bastante obvio lo que hace este código. Más importante aún, el ejemplo muestra las **similitudes sintácticas** entre Pandas y Numpy. En concreto, primero indexamos una columna usando su nombre y luego llamamos a una función usando el **punto** sobre los datos resultantes, es decir, todas las ventas almacenadas en el DataFrame.

In [12]:
# Los datos se leen por URL del repositorio: funciona igual en Colab y en local
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"
df = pd.read_csv(URL + "productos_stock.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/productos_stock.csv'

In [ ]:
# Los datos se leen por URL del repositorio: funciona igual en Colab y en local
URL = "https://raw.githubusercontent.com/Datso653/Laboratorio-de-metodos-Cuantitativos-Aplicados-a-la-gestion/main/DF/"
df = pd.read_csv(URL + "productos_stock.csv")
df.head()

,Producto,Categoría,Stock actual,Precio unitario (€),Ventas mensuales,Días en stock,Proveedor
0,A001,Tecnología,150,450.0,30,60,Delta
1,A002,Tecnología,85,120.0,25,45,Delta
2,B101,Hogar,300,80.0,100,20,HogarTop
3,B102,Hogar,120,60.0,70,25,HogarTop
4,C301,Oficina,200,35.0,60,40,OfficePro


Por qué usar paths relativos (../) en lugar de absolutos:

  1. Portabilidad - El notebook funciona en cualquier máquina sin cambiar nada
  2. Colaboración - Si compartes el código con otros, no necesitan tener la misma estructura 
  de carpetas en C:\Users\pepito\...
  3. Git - Los paths relativos se versionar mejor; los absolutos causan conflictos

In [ ]:
df.info() #muestra la estructura del dataframe


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Producto             10 non-null     object 
 1   Categoría            10 non-null     object 
 2   Stock actual         10 non-null     int64  
 3   Precio unitario (€)  10 non-null     float64
 4   Ventas mensuales     10 non-null     int64  
 5   Días en stock        10 non-null     int64  
 6   Proveedor            10 non-null     object 
dtypes: float64(1), int64(3), object(3)
memory usage: 692.0+ bytes


In [ ]:
df.describe() #resumen estadístico

,Stock actual,Precio unitario (€),Ventas mensuales,Días en stock
count,10.000000,10.000000,10.000000,10.000000
mean,340.500000,83.750000,160.000000,29.500000
std,301.481988,133.806919,189.428966,17.765447
min,85.000000,3.000000,25.000000,10.000000
25%,157.500000,16.250000,48.750000,15.750000
50%,210.000000,42.500000,85.000000,22.500000
75%,375.000000,75.000000,142.500000,43.750000
max,1000.000000,450.000000,600.000000,60.000000


In [ ]:
df["Categoría"].value_counts() #cuenta cuántos productos hay por categoría

,count
Categoría,
Tecnología,2
Hogar,2
Oficina,2
Limpieza,2
Alimentos,2


In [ ]:
df.groupby("Categoría").mean(numeric_only=True) #promedios por categoría

,Stock actual,Precio unitario (€),Ventas mensuales,Días en stock
Categoría,,,,
Alimentos,875.0,3.75,500.0,11.0
Hogar,210.0,70.00,85.0,22.5
Limpieza,310.0,17.50,135.0,16.5
Oficina,190.0,42.50,52.5,45.0
Tecnología,117.5,285.00,27.5,52.5


In [ ]:
df.groupby("Proveedor")["Stock actual"].sum() #stock por proveedor

,Stock actual
Proveedor,
Delta,235
HogarTop,420
LimpioYA,620
OfficePro,380
SanoPlus,1750


In [ ]:
df[df["Stock actual"] < 100]  #productos con bajo stock

,Producto,Categoría,Stock actual,Precio unitario (€),Ventas mensuales,Días en stock,Proveedor
1,A002,Tecnología,85,120.0,25,45,Delta


In [ ]:
# Ejemplos de .iloc (por POSICION/NUMERO) y .loc (por ETIQUETA/NOMBRE)

# CASO 1: .iloc - selecciona por NUMERO de posición
primeros_3_productos = df.iloc[0:3]  # primeras 3 filas (posiciones 0, 1, 2)
print('Primeros 3 productos (iloc):')
print(primeros_3_productos)

# CASO 2: .loc - selecciona por ETIQUETA/NOMBRE (índice)
fila_especifica = df.loc[0]  # fila con índice 0
print('Fila con índice 0 (loc):')
print(fila_especifica)

# CASO 3: .iloc - selecciona columna por número
segunda_columna = df.iloc[:, 1]  # todas las filas, segunda columna (posición 1)
print('Segunda columna (iloc):')
print(segunda_columna.head())

# CASO 4: .loc - selecciona columna por nombre
columna_producto = df.loc[:, 'Producto']  # todas las filas, columna 'Producto'
print('Columna Producto (loc):')
print(columna_producto.head())

# CASO 5: Combinar - filtrar con .loc
bajo_stock = df.loc[df['Stock actual'] < 100, ['Producto', 'Stock actual']]  # donde stock < 100
print('Productos con bajo stock (loc con condición):')
print(bajo_stock)

Primeros 3 productos (iloc):
  Producto   Categoría  Stock actual  Precio unitario (€)  Ventas mensuales  \
0     A001  Tecnología           150                450.0                30   
1     A002  Tecnología            85                120.0                25   
2     B101       Hogar           300                 80.0               100   

   Días en stock Proveedor  
0             60     Delta  
1             45     Delta  
2             20  HogarTop  
Fila con índice 0 (loc):
Producto                     A001
Categoría              Tecnología
Stock actual                  150
Precio unitario (€)         450.0
Ventas mensuales               30
Días en stock                  60
Proveedor                   Delta
Name: 0, dtype: object
Segunda columna (iloc):
0    Tecnología
1    Tecnología
2         Hogar
3         Hogar
4       Oficina
Name: Categoría, dtype: object
Columna Producto (loc):
0    A001
1    A002
2    B101
3    B102
4    C301
Name: Producto, dtype: object
Productos con

## Caso 1
⚡

Supongamos que una empresa evalúa a sus empleados en 3 criterios:

*   Productividad
*   Trabajo en equipo
*   Puntualidad

Cada área (Ventas, Logística, Administración) tiene varios empleados, y queremos calcular el promedio de puntajes por área para compararlas de forma objetiva

In [ ]:
#vamos a representar los puntajes con matrices
#filas: empleados
#columnas: criterios
ventas = np.array([[8, 9, 10],
                   [7, 8, 9],
                   [9, 9, 10]])

logistica = np.array([[6, 7, 8],
                      [7, 6, 7]])

administracion = np.array([[9, 8, 3],
                           [8, 7, 3],
                           [7, 8, 4]])

criterios = ['Productividad', 'Trabajo en equipo', 'Puntualidad']


In [ ]:
#convertimos a dataframe para que se vea bonito y mas sencillo de manejar
df_ventas = pd.DataFrame(ventas, columns=criterios)
df_logistica = pd.DataFrame(logistica, columns=criterios)
df_admin = pd.DataFrame(administracion, columns=criterios)


In [ ]:
#calculamos el promedio por área
prom_ventas = df_ventas.mean()
prom_logistica = df_logistica.mean()
prom_admin = df_admin.mean()

print("Promedios en Ventas:")
print(prom_ventas)
print("\nPromedios en Logística:")
print(prom_logistica)
print("\nPromedios en Administración:")
print(prom_admin)


Promedios en Ventas:
Productividad        8.000000
Trabajo en equipo    8.666667
Puntualidad          9.666667
dtype: float64

Promedios en Logística:
Productividad        6.5
Trabajo en equipo    6.5
Puntualidad          7.5
dtype: float64

Promedios en Administración:
Productividad        8.000000
Trabajo en equipo    7.666667
Puntualidad          3.333333
dtype: float64


## Caso 2
⚡

Una empresa quiere implementar un protocolo para revisar con prioridad las maquinarias que:

* Superan las 3000 horas de uso, y

* Tienen un costo de mantenimiento mayor a 4000 U$D

Tu tarea es ayudar al equipo de gestión a identificar estas maquinarias.

In [ ]:

data = {
    "Nombre": ["Autoelevador A1", "Montacargas X3","Grúa G5", "Carretilla eléctrica R2"],
    "Horas de uso al mes": [400, 100, 89, 250],
    "Mantenimiento (U$/año)": [3500, 2800, 2000, 5000],
    "Estado": ["En servicio", "En servicio", "En servicio", "En servicio"]
}

df_maquinarias = pd.DataFrame(data)
df_maquinarias


,Nombre,Horas de uso al mes,Mantenimiento (U$/año),Estado
0,Autoelevador A1,400,3500,En servicio
1,Montacargas X3,100,2800,En servicio
2,Grúa G5,89,2000,En servicio
3,Carretilla eléctrica R2,250,5000,En servicio


In [ ]:
#creamos la columna de rendimiento, es un ratio una relacion entre cantidades
df_maquinarias["Rendimiento"] = df_maquinarias["Horas de uso al mes"] / df_maquinarias["Mantenimiento (U$/año)"]


In [ ]:
#clasificamos el rendimiento
condiciones = [
    (df_maquinarias["Rendimiento"] > 0.1),
    (df_maquinarias["Rendimiento"] > 0.05) & (df_maquinarias["Rendimiento"] <= 0.1),
    (df_maquinarias["Rendimiento"] <= 0.05)
]
clasificaciones = ["Alto", "Medio", "Bajo"]
df_maquinarias["Clasificación"] = np.select(condiciones, clasificaciones,default="Sin datos")

In [ ]:
#filtramos las de bajo rendimiento
bajo_rendimiento = df_maquinarias[df_maquinarias["Clasificación"] == "Bajo"]
bajo_rendimiento


,Nombre,Horas de uso al mes,Mantenimiento (U$/año),Estado,Rendimiento,Clasificación
1,Montacargas X3,100,2800,En servicio,0.035714,Bajo
2,Grúa G5,89,2000,En servicio,0.044500,Bajo
3,Carretilla eléctrica R2,250,5000,En servicio,0.050000,Bajo


---
## 🧱 Matrices especiales: cada forma modela algo distinto

No son curiosidades matemáticas. **La estructura de una matriz cuenta cómo funciona el problema:**

| Estructura | Qué modela | Ejemplo de gestión |
|---|---|---|
| **Identidad** | Ningún efecto cruzado, escala 1 | Punto de comparación |
| **Diagonal** | Efectos independientes, sin interacción | Una merma distinta por producto |
| **Triangular** | Dependencia **con orden** | Etapas de producción: corte → costura → terminado |
| **Simétrica** | Relación **recíproca** | Distancias entre plantas |
| **Densa** (sin estructura) | Todo afecta a todo | Requerimientos de recursos |

> 🎯 Cuando te dan una matriz, **mirá su forma antes de calcular**. Si es triangular, hay un orden.
> Si es simétrica, la relación va en los dos sentidos. Eso ya te dice algo del negocio.

In [ ]:
I = np.eye(3)                              # identidad
mermas = np.diag([1.03, 1.04, 1.06])       # diagonal: una merma por producto
distancias = np.array([[  0, 520, 680],    # simétrica: la distancia va en los dos sentidos
                       [520,   0, 410],
                       [680, 410,   0]])

print("Identidad:\n", I)
print("\nDiagonal de mermas:\n", mermas)
print("\nDistancias entre plantas:\n", distancias)

### Verificar una propiedad, no suponerla

Antes de confiar en que una matriz "es simétrica" o "es diagonal", **comprobalo**:

| Propiedad | Qué significa | Cómo se verifica |
|---|---|---|
| Identidad | $A \cdot I = A$ | `np.allclose(A @ I, A)` |
| Simétrica | $M = M^T$ | `np.allclose(M, M.T)` |
| Diagonal | Todo lo de afuera es cero | `np.allclose(A, np.diag(np.diag(A)))` |

> ⚠️ **Nunca compares matrices con `==`.** Eso devuelve una matriz de `True`/`False`, no una
> respuesta. Y con decimales falla por errores de redondeo. Usá siempre **`np.allclose`**.

In [ ]:
def describir(nombre, M):
    """Verifica las propiedades de una matriz y devuelve una conclusión, no una matriz."""
    print(f"{nombre}:")
    print("   ¿simétrica? ", np.allclose(M, M.T))
    print("   ¿diagonal?  ", np.allclose(M, np.diag(np.diag(M))))

describir("Distancias", distancias)
describir("Mermas", mermas)
describir("Ventas", ventas)

# Por qué no usar == :
print("\nventas == ventas.T devuelve esto (una matriz, no una respuesta):")
print(ventas == ventas.T)

# 🥇 ⚡ 🤓  Para tirar una matriz random

In [ ]:
def generar_matriz():
    #tamaño de la matriz
    while True:
        try:
            filas = int(input("Ingrese número de filas : "))
            columnas = int(input("Ingrese número de columnas : "))
            if filas > 0 and columnas > 0:
                break
            else:
                print("Deben ser números enteros positivos.")
        except ValueError:
            print("Entrada inválida. Ingrese enteros positivos.")

    #rango de valores
    while True:
        try:
            mínimo = float(input("Ingrese el valor mínimo: "))
            máximo = float(input("Ingrese el valor máximo: "))
            if mínimo <= máximo:
                break
            else:
                print("El mínimo no puede ser mayor que el máximo.")
        except ValueError:
            print("Entrada inválida. Ingrese números.")

    #tipo de número
    while True:
        tipo = input("¿Desea números enteros (E) o decimales (D)? ").strip().lower()
        if tipo in ["e", "d"]:
            break
        else:
            print("⚠️ Escriba 'E' para enteros o 'D' para decimales.")

    #generar matriz
    if tipo == "e":
        matriz = np.random.randint(int(mínimo), int(máximo) + 1, size=(filas, columnas))
    else:
        matriz = np.random.uniform(mínimo, máximo, size=(filas, columnas))

    print("\n✅ Matriz generada:")
    print(matriz)

    return matriz

#entonces por ejemplo
M = generar_matriz()



✅ Matriz generada:
[[3 5 2]
 [3 3 4]
 [4 4 4]]


-Actividad propuesta

1.   Crear una matriz de producción para una Economía hipotética. Pueden agregar la cantidad de sectores que deseen
2.   Convertirla en una tabla de datos
3.   Incorporar una fila que muestre el valor agregado por sector, es decir, la diferencia entre el producto total y la suma del valor del sector.
4.   Suponer un cambio en la Demanda Final de cada sector y calcular cómo varía el Producto total en consecuencia.


Fuentes:

Notas de álgebra teórico-prácticas: cátedra de Álgebra / Alicia Delia Fraquelli;Andrea Leonor Gache. - 1a ed. - Ciudad Autónoma de Buenos Aires: Universidad de Buenos Aires. Facultad de Ciencias Económicas, 2019.

Hilpisch, Y. (2018). Python for Finance: Mastering Data-Driven Finance (2nd ed.). O’Reilly Media.

---
## 🧭 Para llevarse

| Concepto | Comando |
|---|---|
| Crear un array | `np.array(lista)` |
| Matriz de ceros / unos / identidad | `np.zeros((f,c))` · `np.ones((f,c))` · `np.eye(n)` |
| Forma de la tabla | `A.shape` → `(filas, columnas)` |
| Total por columna / por fila | `A.sum(axis=0)` · `A.sum(axis=1)` |
| Multiplicar celda a celda | `A * B` |
| **Producto matricial** | `A @ B` |
| Transponer | `A.T` |
| Diagonal | `np.diag(v)` |
| Comparar matrices | `np.allclose(A, B)` — **nunca `==`** |

**Las tres ideas de la clase:**
1. **Vectorizar** es decir qué hacer con todos los datos, no cómo recorrerlos.
2. `axis` indica **qué eje desaparece**: `axis=0` colapsa filas, `axis=1` colapsa columnas.
3. `*` y `@` son operaciones **distintas** y ninguna avisa cuando elegiste mal.

En la próxima clase usamos todo esto para resolver sistemas de ecuaciones.